In [1]:
from rag.parser.parser import Parser
from rag.utils.file_validator import FileInspector
from pathlib import Path
from fastembed import TextEmbedding
from rag.parser.code_parser import CodeParser
import json
from copy import deepcopy
from typing import Any

# from transformers import AutoTokenizer
# from tokenizers

/home/user/Documents/Project_5/backend/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
# bge-small's real limit -- use the actual tokenizer, not a line-count guess.
# Line count is a bad proxy: a minified 3-line function and a readable
# 3-line function can differ by 10x in actual token count.
MODEL_NAME = "BAAI/bge-small-en-v1.5"

_MODEL = TextEmbedding(model_name=MODEL_NAME, threads=1)
TOKEN_LIMIT = 480  # small safety margin under the real 512 limit

# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def count_tokens(text: str) -> int:
    return _MODEL.token_count(text)

In [2]:
root_path = Path("/home/user/Documents/Project_5/backend/experiments/data/code/documents/spreadsheet")

In [8]:
print(count_tokens("The diagram presents the architecture of a chat application. A user interacts with the frontend through a web or mobile interface. The frontend sends chat requests to a backend API, which handles authentication, validation, conversation management, and message processing. The backend retrieves previous messages from a database and sends the current prompt to an AI model or external model provider. The generated response is returned to the backend, stored as part of the conversation history, and then sent back to the frontend for display. Additional components may include file storage, an embedding service, a vector database for document search, and monitoring tools. Arrows show how requests and responses move between the user interface, backend services, databases, and AI provider."))

151


In [2]:
inspector = FileInspector()
codeParser = CodeParser()
parser = Parser()

In [4]:
from rag.parser.document_parser import DocumentParser
from rag.parser.markdown_parser import MarkDownParser
from rag.chunker.markdown_chunker import MarkdownChunker
from rag.embedder.embedder import Embedder

doc_parser = DocumentParser()
md_parser = MarkDownParser()
em = Embedder()

all_chunks = []

for file_path in root_path.rglob("*"):
    if not file_path.is_file():
        continue

    inspector.set_file_path(file_path)
    meta = inspector.inspect()
    meta["file"] = file_path

    print(meta)

    result = parser.parse_csv_and_spreadsheets(
        file_name=file_path,
        language=file_path.suffix.lower(),
    )

    if result is not None:
        all_chunks.append(result)
        # print(chunks[-1])

            # markdown_text = doc_parser.parse(file_path)
            # md_parser._set_file_path(file_path)
            # parsed = md_parser.parser(doc=markdown_text, file_path=None)
            # print(markdown_text)
            # print(parsed)


# plain_text = re.sub(r"\\[a-z]+\d* ?", "", rtf_content)
# plain_text = re.sub(r"[{}]", "", plain_text)

# print(plain_text)
# markdown_text = doc_parser.parse(
#     Path(
#         "/home/user/Documents/Project_5/backend/experiments/data/code/file.rtf"
#     )
# )
# # print(markdown_text)
# md_c = MarkdownChunker(
#     file_path=Path(
#         "/home/user/Documents/Project_5/backend/experiments/data/code/file.rtf"
#     ),
#     ext_lang_name=parser.extension_to_language_name,
# )
# parsed = md_parser.parser(doc=markdown_text, file_path=None, use_vision_llm=False)
# # print(markdown_text)
# chunks = md_c._merge_rag_chunks(chunks=parsed, count_token=em.count_tokens)

# data = []
# for chunk in chunks:
#     em_data = {"content": chunk, "tokens": em.count_tokens(chunk)}
#     data.append(em_data)
# chunk = parser.parse_csv_and_spreadsheats(file_name=Path("/home/user/Documents/Project_5/backend/experiments/data/code/documents/spreadsheet/test.ods"), language=".ods")
print(all_chunks)

{'category': 'document', 'format': 'csv', 'mime': 'text/csv', 'should_parse': False, 'file': PosixPath('/home/user/Documents/Project_5/backend/experiments/data/code/documents/spreadsheet/test.xlsb')}
Not a supported CSV or spreadsheet file: .xlsb
{'category': 'document', 'format': 'tsv', 'mime': 'text/plain', 'should_parse': False, 'file': PosixPath('/home/user/Documents/Project_5/backend/experiments/data/code/documents/spreadsheet/test.tsv')}
{'category': 'document', 'format': 'csv', 'mime': 'text/csv', 'should_parse': False, 'file': PosixPath('/home/user/Documents/Project_5/backend/experiments/data/code/documents/spreadsheet/olum.csv')}
{'category': 'text', 'mime': 'text/xml', 'language_alias': 'unknown', 'language_name': 'unknown', 'should_parse': False, 'file': PosixPath('/home/user/Documents/Project_5/backend/experiments/data/code/documents/spreadsheet/test.fods')}
Not a supported CSV or spreadsheet file: .fods
{'category': 'document', 'format': 'ods', 'mime': 'application/vnd.oas

In [4]:
inspector.set_file_path(Path("/home/user/Documents/Project_5/backend/experiments/data/code/go.mod"))
print(inspector.inspect())

{'category': 'code', 'mime': 'inode/x-empty', 'language_alias': 'modula2', 'language_name': 'Modula-2', 'should_parse': True}


In [ ]:
data = []
complete_data = []
language_alias = ["java", "go", "javascript", "typescript", "python"]
has_read = False
for file_path in root_path.rglob("*"):
    chunks = []
    if file_path.is_file():
        inspector.set_file_path(file_path)
        meta = inspector.inspect()
        if meta.get("language_alias") in language_alias:
            # has_read = True
            codeParser._set_file_path(file_path)
            tree = codeParser.parse_ast()
            if tree:
                # print(tree)
                source_bytes = codeParser._get_source_bytes()
                if not source_bytes:
                    continue
                chunks = codeParser.extract_chunks(tree.root_node)
        for chunk in chunks:
            complete_data.append({"file_path": file_path, "chunk": chunk})
            searilizable_chunk = {k: v for k, v in chunk.items() if k != "_node"}
            data.append(
                {
                    "file_path": str(file_path),
                    "file_name": file_path.name,
                    "chunk": chunk,
                    "len": count_tokens(json.dumps(searilizable_chunk)),
                }
            )
print(data)

[{'file_path': '/home/user/Documents/Project_5/backend/experiments/data/code/palindrome.java', 'file_name': 'palindrome.java', 'chunk': {'_node': <Node type=class_declaration, start_point=(0, 0), end_point=(18, 1)>, 'kind': 'class_summary', 'name': 'Palindrome', 'comment': None, 'content': 'class Palindrome: methods = main', 'start_line': 0, 'end_line': 18}, 'len': 62}, {'file_path': '/home/user/Documents/Project_5/backend/experiments/data/code/palindrome.java', 'file_name': 'palindrome.java', 'chunk': {'_node': <Node type=method_declaration, start_point=(1, 4), end_point=(17, 5)>, 'kind': 'function', 'name': 'main', 'comment': None, 'parent_class': 'Palindrome', 'parent_function': None, 'content': 'public static void main(String[] args) {\n        int num = 121;\n        int n = num;\n        int reversed = 0;\n\n        while (n > 0) {\n            int digit = n % 10;\n            reversed = reversed * 10 + digit;\n            n = n / 10;\n        }\n\n        if (num == reversed) {\

In [8]:
sorted_data = sorted(data, key=lambda x: x["len"], reverse=True)

print(sorted_data)

[{'file_path': '/home/user/Documents/Project_5/backend/experiments/data/code/bank.go', 'file_name': 'bank.go', 'chunk': {'kind': 'function', 'name': 'main', 'comment': '// ==========================================\n// MAIN FUNCTION (Demonstration)\n// ==========================================', 'parent_class': None, 'parent_function': None, 'content': 'func main() {\n\tfmt.Println("=== Object-Oriented Programming (OOP) in Go Demo ===\\n")\n\n\t// Creating objects using the interface to demonstrate Polymorphism\n\tvar acc1 BankAccount = NewSavingsAccount("SA101", "Alice Smith", 1000.0, 4.5)\n\tvar acc2 BankAccount = NewCheckingAccount("CA201", "Bob Jones", 500.0, 200.0)\n\n\tfmt.Println("--- 1. Savings Account Operations ---")\n\tfmt.Printf("Holder: %s | Initial Balance: $%.1f\\n", acc1.GetAccountHolder(), acc1.GetBalance())\n\tacc1.Deposit(200.0)\n\tacc1.CalculateInterest() // Polymorphic call\n\tacc1.Withdraw(1200.0)    // Should fail due to $50 minimum balance rule\n\tacc1.Withdraw

In [9]:
def create_embeding_text(chunk: dict) -> str:
    parts = []

    kind = chunk.get("kind")
    name = chunk.get("name")
    comment = chunk.get("comment")
    content = chunk.get("content")
    parent_class = chunk.get("parent_class")
    parent_function = chunk.get("parent_function")

    parts.append(f"File: {path.name} | Language: {path.suffix}")

    if kind is not None:
        parts.append(f"Kind: {kind}")

    if name is not None:
        parts.append(f"Name: {name}")

    if parent_class is not None:
        parts.append(f"Parent class: {parent_class}")

    if parent_function is not None:
        parts.append(f"Parent function: {parent_function}")

    code_parts = []

    if comment is not None:
        code_parts.append(str(comment))

    if content is not None:
        code_parts.append(str(content))

    if code_parts:
        parts.append("Code: " + "\n".join(code_parts))

    # Use append, not extend
    text_to_embed = " | ".join(parts)
    return text_to_embed

In [ ]:
def split_oversized_chunk(
    chunk: dict, source_bytes: bytes, token_limit: int = 480
) -> list[dict]:
    clean_chunk = {k: v for k, v in chunk.items() if k != "_node"}

    # 1. Base Check: Does it already fit?
    # print(count_tokens(create_embeding_text(clean_chunk)))
    if count_tokens(create_embeding_text(clean_chunk)) <= token_limit:
        return [clean_chunk]

    node = chunk.get("_node")
    sub_chunks = []

    # 2. Try Tree-Sitter Statement-Level Splitting
    if node:
        body = node.child_by_field_name("body") or node
        statements = [c for c in body.children if c.is_named]

        signature = ""
        if node.type in ["function_declaration", "method_declaration"]:
            signature = source_bytes[node.start_byte : body.start_byte].decode("utf-8")

        current_group = []
        part = 1

        def create_ts_candidate(group, p):
            start = group[0].start_byte
            end = group[-1].end_byte
            text = signature + source_bytes[start:end].decode("utf-8")
            if signature:
                text += "\n}"
            return {
                **clean_chunk,
                "content": text,
                "name": f"{clean_chunk.get('name', 'unnamed')} (part {p})",
                "parent_chunk_id": clean_chunk.get("id"),
                "start_line": group[0].start_point[0],
                "end_line": group[-1].end_point[0],
            }

        for stmt in statements:
            test_group = current_group + [stmt]
            candidate = create_ts_candidate(test_group, part)

            if count_tokens(json.dumps(candidate)) > token_limit and current_group:
                sub_chunks.append(create_ts_candidate(current_group, part))
                part += 1
                current_group = [stmt]
            else:
                current_group.append(stmt)

        if current_group:
            sub_chunks.append(create_ts_candidate(current_group, part))

    # 3. Validation & Text Fallback
    # If _node was missing, or Tree-sitter returned a single statement still over the limit,
    # we force a line-by-line split.
    needs_fallback = False
    if not sub_chunks:
        needs_fallback = True
    else:
        for sc in sub_chunks:
            if count_tokens(json.dumps(sc)) > token_limit:
                needs_fallback = True
                break

    if needs_fallback:
        sub_chunks = []
        lines = clean_chunk["content"].split("\n")
        current_lines = []
        part = 1

        for line in lines:
            test_lines = current_lines + [line]
            candidate_text = "\n".join(test_lines)

            candidate = {
                **clean_chunk,
                "content": candidate_text,
                "name": f"{clean_chunk.get('name', 'unnamed')} (part {part})",
                "parent_chunk_id": clean_chunk.get("id"),
            }

            if (
                count_tokens(create_embeding_text(candidate)) > token_limit
                and current_lines
            ):
                # Flush previous lines
                sub_chunks.append(
                    {
                        **clean_chunk,
                        "content": "\n".join(current_lines),
                        "name": f"{clean_chunk.get('name', 'unnamed')} (part {part})",
                        "parent_chunk_id": clean_chunk.get("id"),
                    }
                )
                part += 1
                current_lines = [line]
            else:
                current_lines.append(line)

        if current_lines:
            sub_chunks.append(
                {
                    **clean_chunk,
                    "content": "\n".join(current_lines),
                    "name": f"{clean_chunk.get('name', 'unnamed')} (part {part})",
                    "parent_chunk_id": clean_chunk.get("id"),
                }
            )

    return sub_chunks

In [11]:
from pathlib import Path

content_to_embed = []

for data in complete_data:
    file_path = data.get("file_path")

    if not file_path:
        continue

    # Convert the path to a Path object if it is stored as a string
    path = Path(file_path)

    if not path.is_file():
        print(f"File not found: {path}")
        continue

    source_bytes = path.read_bytes()

    chunks = split_oversized_chunk(data.get("chunk"), source_bytes)

    for chunk in chunks:
        text_to_embed = create_embeding_text(chunk)
        content_to_embed.append(
            {"content": text_to_embed, "tokens": count_tokens(text_to_embed)}
        )

print(content_to_embed)

[{'content': 'File: palindrome.java | Language: .java | Kind: class_summary | Name: Palindrome | Code: class Palindrome: methods = main', 'tokens': 40}, {'content': 'File: palindrome.java | Language: .java | Kind: function | Name: main | Parent class: Palindrome | Code: public static void main(String[] args) {\n        int num = 121;\n        int n = num;\n        int reversed = 0;\n\n        while (n > 0) {\n            int digit = n % 10;\n            reversed = reversed * 10 + digit;\n            n = n / 10;\n        }\n\n        if (num == reversed) {\n            System.out.println(num + " is a palindrome");\n        } else {\n            System.out.println(num + " is not a palindrome");\n        }\n    }', 'tokens': 147}, {'content': 'File: palindrome.go | Language: .go | Kind: function | Name: main | Code: func main() {\n\tnum := 121\n\tn := num\n\treversed := 0\n\n\tfor n > 0 {\n\t\tdigit := n % 10\n\t\treversed = reversed*10 + digit\n\t\tn /= 10\n\t}\n\n\tif num == reversed {\

In [12]:
# data = []
# for chunk in content_to_embed:
#     # print(chunk)
#     chunk_str = str(chunk)
#     data.append({"content": chunk_str, "size": count_tokens(chunk_str)})

sorted_data = sorted(content_to_embed, key=lambda x: x["tokens"], reverse=True)
print(sorted_data)

[{'content': 'File: bank.js | Language: .js | Kind: function | Name: main | Code: // ==========================================\n// MAIN FUNCTION (Demonstration)\n// ==========================================\n/* \n    THIS METHOD CALLS EACH AND EVERY METHOD\n    CREATES NEW INSTANCE OF EACH OF THE CLASS\n*/\nfunction main() {\n    console.log("=== Object-Oriented Programming (OOP) in JavaScript Demo ===\\n");\n\n    // Creating objects using polymorphism\n    const acc1 = new SavingsAccount("SA101", "Alice Smith", 1000.0, 4.5);\n    const acc2 = new CheckingAccount("CA201", "Bob Jones", 500.0, 200.0);\n\n    console.log("--- 1. Savings Account Operations ---");\n    console.log(`Holder: ${acc1.getAccountHolder()} | Initial Balance: $${acc1.getBalance()}`);\n    acc1.deposit(200.0);\n    acc1.calculateInterest(); // Polymorphic call\n    acc1.withdraw(1200.0);    // Should fail due to $50 minimum balance rule\n    acc1.withdraw(100.0);     // Should succeed\n\n    console.log("\\n--- 2

In [ ]:
data_to_embed = [data["content"] for data in sorted_data]
node_embedding = list(_MODEL.embed(data_to_embed, parallel=1))

node_embedding[0]

In [41]:
from IPython.display import display, HTML

In [28]:
import networkx as nx
from pyvis.network import Network
import matplotlib.pyplot as plt
import umap

In [19]:
reducer = umap.UMAP(n_components=2, random_state=42)
embeddings_2d = reducer.fit_transform(node_embedding)

/home/user/Documents/Project_5/backend/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [ ]:
import numpy as np
import networkx as nx
import umap
import plotly.graph_objects as go
from sklearn.neighbors import NearestNeighbors

# 1. Dimensionality reduction for visualization coordinates
reducer = umap.UMAP(n_components=2, random_state=42)
coords = reducer.fit_transform(node_embedding)

# 2. Build Graph using k-NN instead of a global similarity threshold
K_NEIGHBORS = 3  # Connect each node only to its top 3 closest matches

nn = NearestNeighbors(n_neighbors=K_NEIGHBORS + 1, metric="cosine")
nn.fit(node_embedding)
distances, indices = nn.kneighbors(node_embedding)

G = nx.Graph()
node_labels = [f"file_{i}.txt" for i in range(len(node_embedding))]

for i in range(len(node_labels)):
    G.add_node(i, label=node_labels[i])
    for j in range(1, K_NEIGHBORS + 1):  # Skip index 0 (self-match)
        neighbor_idx = indices[i][j]
        G.add_edge(i, neighbor_idx)

# 3. Build Edge Traces
edge_x, edge_y = [], []
for u, v in G.edges():
    x0, y0 = coords[u]
    x1, y1 = coords[v]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])

edge_trace = go.Scatter(
    x=edge_x,
    y=edge_y,
    line=dict(width=0.6, color="#aaa"),
    hoverinfo="none",
    mode="lines",
)

# 4. Build Node Traces
node_x = [coords[i][0] for i in G.nodes()]
node_y = [coords[i][1] for i in G.nodes()]
node_degrees = [deg for _, deg in G.degree()]
node_hover = [
    f"File: {G.nodes[i]['label']}<br>Connections: {deg}" for i, deg in G.degree()
]

node_trace = go.Scatter(
    x=node_x,
    y=node_y,
    mode="markers",
    hoverinfo="text",
    text=node_hover,
    marker=dict(
        showscale=True,
        colorscale="YlGnBu",
        reversescale=True,
        color=node_degrees,
        size=8,
        colorbar=dict(thickness=15, title=dict(text="Connections")),
    ),
)

# 5. Render Figure
fig = go.Figure(
    data=[edge_trace, node_trace],
    layout=go.Layout(
        showlegend=False,
        hovermode="closest",
        margin=dict(b=20, l=5, r=5, t=40),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    ),
)

fig.show(renderer="browser")

/home/user/Documents/Project_5/backend/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [6]:
from rag.parser.markdown_parser import MarkDownParser
from rag.chunker.markdown_chunker import MarkdownChunker
from rag.embedder.embedder import Embedder
from pathlib import Path

In [7]:
markdown_file = Path(
    "/home/user/Documents/Project_5/backend/experiments/data/code/documents/test.md"
)

In [8]:
emd = Embedder()
md_parser = MarkDownParser()
md_chunker = MarkdownChunker(markdown_file, inspector.extension_to_language_name)
md_parser._set_file_path(markdown_file)

In [9]:
ast = md_parser.parse_ast()
if ast is not None:
    raw_chunks = md_parser._parse_markdown(ast.root_node)
    chunks = md_chunker._merge_rag_chunks(raw_chunks, emd.count_tokens)
    data = md_chunker.convert_all_to_embeding_text(chunks)
    converted = []
    for d in data:
        tokens = count_tokens(d)
        converted.append({"data": d, "tokens": tokens})
    print(chunks)

HTML image found in paragraph: This is the html image of the architecure ... Yo Yo
<img src="https://media.geeksforgeeks.org/wp-content/uploads/20250304153405016345/Deployment-Architecture-Diagram.webp" alt="Application Screenshot" width="500">

[{'content': '# Python and Artificial Intelligence (AI)\n\n\n<h2>THis is HTML Embeding</h2>\n\n\n> **Python** is a high-level, general-purpose programming language known for its simple syntax, readability, and large ecosystem of libraries.\n\n\n---\n\n## 1. What Is Python?\n\nPython is a programming language created by **Guido van Rossum** and first released in **1991**.', 'token_count': 226, 'metadata': {'headings': {1: '# Python and Artificial Intelligence (AI)\n'}, 'sections_covered': {'h1': ['# Python and Artificial Intelligence (AI)\n'], 'h2': ['## 1. What Is Python?\n']}, 'is_image': False, 'is_table': False, 'contains_code_block': False}}, {'content': 'It is designed to make programming easier to read and write. Python uses indentation t

In [10]:
print(converted)

[{'data': "File: /home/user/Documents/Project_5/backend/experiments/data/code/documents/test.md\n\nLanguage: ('markdown', 'Markdown')\n\nContent type: text\n\nSection context:\nH1: # Python and Artificial Intelligence (AI)\n\nSections covered:\nH1: # Python and Artificial Intelligence (AI)\nH2: ## 1. What Is Python?\n\nContent:\n# Python and Artificial Intelligence (AI)\n\n\n<h2>THis is HTML Embeding</h2>\n\n\n> **Python** is a high-level, general-purpose programming language known for its simple syntax, readability, and large ecosystem of libraries.\n\n\n---\n\n## 1. What Is Python?\n\nPython is a programming language created by **Guido van Rossum** and first released in **1991**.", 'tokens': 178}, {'data': "File: /home/user/Documents/Project_5/backend/experiments/data/code/documents/test.md\n\nLanguage: ('markdown', 'Markdown')\n\nContent type: text\n\nSection context:\nH1: # Python and Artificial Intelligence (AI)\nH2: ## 1. What Is Python?\n\nSections covered:\nH1: # Python and Art

[{'content': '# Python and Artificial Intelligence (AI)\n\n\n<h2>THis is HTML Embeding</h2>\n\n\n> **Python** is a high-level, general-purpose programming language known for its simple syntax, readability, and large ecosystem of libraries.\n\n\n---\n\n## 1. What Is Python?\n\n\nPython is a programming language created by **Guido van Rossum** and first released in **1991**.\n\n\nIt is designed to make programming easier to read and write. Python uses indentation to define blocks of code instead of relying heavily on symbols such as `{}`.', 'word_count': 260, 'metadata': {'headings': {1: '# Python and Artificial Intelligence (AI)\n'}, 'sections_covered': {'h1': ['# Python and Artificial Intelligence (AI)\n'], 'h2': ['## 1. What Is Python?\n']}, 'is_image': False, 'is_table': False, 'contains_code_block': False}}, {'metadata': {'headings': {1: '# Python and Artificial Intelligence (AI)\n', 2: '## Architecture\n'}, 'is_image': True, 'is_code_block': False, 'is_table': False, 'image': {'ima

In [6]:
from rag.parser.parser import Parser
from rag.embedder.embedder import Embedder
from pathlib import Path
parser = Parser()
embedder = Embedder()

Fetching 5 files: 100%|██████████| 5/5 [00:14<00:00,  2.81s/it]


In [8]:
parser._set_file_path(file_path=Path("/home/user/Documents/Project_5/backend/experiments/data/code/package.json"))

In [10]:
print(parser.general_parser(embedder.count_tokens, target=450))

[{'content': '{ "name": "front-end", "version": "0.1.0", "private": true, "scripts": { "dev": "next dev --turbopack", "build": "next build --turbopack", "start": "next start", "lint": "eslint" }, "dependencies": { "@dnd-kit/core": "^6.3.1", "@dnd-kit/modifiers": "^9.0.0", "@dnd-kit/sortable": "^10.0.0", "@dnd-kit/utilities": "^3.2.2", "@hookform/resolvers": "^5.2.2", "@radix-ui/react-accordion": "^1.2.12", "@radix-ui/react-avatar": "^1.1.11", "@radix-ui/react-checkbox": "^1.3.3", "@radix-ui/react-dialog": "^1.1.15", "@radix-ui/react-dropdown-menu": "^2.1.16", "@radix-ui/react-label": "^2.1.8", "@radix-ui/react-radio-group": "^1.3.8", "@radix-ui/react-select": "^2.2.6", "@radix-ui/react-separator": "^1.1.8", "@radix-ui/react-slot": "^1.2.4", "@radix-ui/react-switch": "^1.2.6", "@radix-ui/react-tabs":', 'metadata': {'word_count': 58, 'token_count': 450, 'file_name': 'package.json', 'file_path': '/home/user/Documents/Project_5/backend/experiments/data/code/package.json', 'file_type': 'jso

In [ ]:
[
    {
        "content": '{ "name": "front-end", "version": "0.1.0", "private": true, "scripts": { "dev": "next dev --turbopack", "build": "next build --turbopack", "start": "next start", "lint": "eslint" }, "dependencies": { "@dnd-kit/core": "^6.3.1", "@dnd-kit/modifiers": "^9.0.0", "@dnd-kit/sortable": "^10.0.0", "@dnd-kit/utilities": "^3.2.2", "@hookform/resolvers": "^5.2.2", "@radix-ui/react-accordion": "^1.2.12", "@radix-ui/react-avatar": "^1.1.11", "@radix-ui/react-checkbox": "^1.3.3", "@radix-ui/react-dialog": "^1.1.15", "@radix-ui/react-dropdown-menu": "^2.1.16", "@radix-ui/react-label": "^2.1.8", "@radix-ui/react-radio-group": "^1.3.8", "@radix-ui/react-select": "^2.2.6", "@radix-ui/react-separator": "^1.1.8", "@radix-ui/react-slot": "^1.2.4", "@radix-ui/react-switch": "^1.2.6", "@radix-ui/react-tabs":',
        "metadata": {
            "word_count": 58,
            "token_count": 450,
            "file_name": "package.json",
            "file_path": "/home/user/Documents/Project_5/backend/experiments/data/code/package.json",
            "file_type": "json",
        },
    },
    {
        "content": '"@radix-ui/react-slot": "^1.2.4", "@radix-ui/react-switch": "^1.2.6", "@radix-ui/react-tabs": "^1.1.13", "@radix-ui/react-toggle": "^1.1.10", "@radix-ui/react-toggle-group": "^1.1.11", "@radix-ui/react-tooltip": "^1.2.8", "@reduxjs/toolkit": "^2.11.2", "@supabase/ssr": "^0.7.0", "@supabase/supabase-js": "^2.75.1", "@tabler/icons-react": "^3.35.0", "@tanstack/react-table": "^8.21.3", "@tiptap/extension-bubble-menu": "^3.9.1", "@tiptap/extension-code-block-lowlight": "^3.9.1", "@tiptap/extension-highlight": "^3.9.1", "@tiptap/extension-image": "^3.9.1", "@tiptap/extension-subscript": "^3.9.1", "@tiptap/extension-superscript": "^3.9.1", "@tiptap/extension-text-align": "^3.9.1", "@tiptap/extension-text-style": "^3.9.1", "@tiptap/extension-youtube": "^3.9.1", "@tiptap/pm": "^3.9.1",',
        "metadata": {
            "word_count": 42,
            "token_count": 450,
            "file_name": "package.json",
            "file_path": "/home/user/Documents/Project_5/backend/experiments/data/code/package.json",
            "file_type": "json",
        },
    },
    {
        "content": '"@tiptap/extension-text-style": "^3.9.1", "@tiptap/extension-youtube": "^3.9.1", "@tiptap/pm": "^3.9.1", "@tiptap/react": "^3.9.1", "@tiptap/starter-kit": "^3.9.1", "class-variance-authority": "^0.7.1", "clsx": "^2.1.1", "dompurify": "^3.3.0", "lowlight": "^3.3.0", "lucide-react": "^0.546.0", "next": "15.5.6", "next-themes": "^0.4.6", "react": "19.1.0", "react-dom": "19.1.0", "react-hook-form": "^7.65.0", "react-redux": "^9.2.0", "recharts": "^2.15.4", "sonner": "^2.0.7", "tailwind-merge": "^3.3.1", "vaul": "^1.1.2", "zod": "^4.2.1" }, "devDependencies": { "@eslint/eslintrc": "^3", "@tailwindcss/postcss": "^4", "@types/node": "^20", "@types/react": "^19", "@types/react-dom": "^19", "eslint": "^9", "eslint-config-next": "15.5.6",',
        "metadata": {
            "word_count": 59,
            "token_count": 450,
            "file_name": "package.json",
            "file_path": "/home/user/Documents/Project_5/backend/experiments/data/code/package.json",
            "file_type": "json",
        },
    },
    {
        "content": '"^19", "@types/react-dom": "^19", "eslint": "^9", "eslint-config-next": "15.5.6", "tailwindcss": "^4", "tw-animate-css": "^1.4.0", "typescript": "^5" } }',
        "metadata": {
            "word_count": 15,
            "token_count": 95,
            "file_name": "package.json",
            "file_path": "/home/user/Documents/Project_5/backend/experiments/data/code/package.json",
            "file_type": "json",
        },
    },
]